In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from global_parameters import Assumptions

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

# Creating combinations of planforms

In [2]:
span_max = 2.7 #TODO change

lists_to_recombine = dict()

lists_to_recombine['aspect_ratio'] = [5., 10., 17., 27.]
lists_to_recombine['taper'] = [1.]
lists_to_recombine['thickness_to_chord'] = [.12, .15, .18] #NOTE not super justified
lists_to_recombine['sweep'] = [-10., 15.] #NOTE not super justified
lists_to_recombine['cl_alpha'] = [2*np.pi]
lists_to_recombine['cl_max'] = [1.]
lists_to_recombine['cm_ac'] = [-.15, 0.1] #TODO change
lists_to_recombine['cl_0'] = [0.]
lists_to_recombine['pf_type'] = ['tail', 'canard']

In [3]:
ltr_keys = lists_to_recombine.keys()
ltr_values = lists_to_recombine.values()

planforms_raw = list(itt.product(*ltr_values))
print(planforms_raw)

planform_params = list()
for planform_raw in planforms_raw:
    planform_param = dict()
    for i, key in enumerate(ltr_keys):
        planform_param[key] = planform_raw[i]
    planform_params.append(planform_param)

assert len(planform_params) == 192//2, len(planform_params)

[(5.0, 1.0, 0.12, -10.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.12, -10.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.12, -10.0, 6.283185307179586, 1.0, 0.1, 0.0, 'tail'), (5.0, 1.0, 0.12, -10.0, 6.283185307179586, 1.0, 0.1, 0.0, 'canard'), (5.0, 1.0, 0.12, 15.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.12, 15.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.12, 15.0, 6.283185307179586, 1.0, 0.1, 0.0, 'tail'), (5.0, 1.0, 0.12, 15.0, 6.283185307179586, 1.0, 0.1, 0.0, 'canard'), (5.0, 1.0, 0.15, -10.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.15, -10.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.15, -10.0, 6.283185307179586, 1.0, 0.1, 0.0, 'tail'), (5.0, 1.0, 0.15, -10.0, 6.283185307179586, 1.0, 0.1, 0.0, 'canard'), (5.0, 1.0, 0.15, 15.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.15, 15.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.15, 15.0, 6.283185307179

In [4]:
wing_area = span_max**2/max(lists_to_recombine['aspect_ratio'])

planforms:list[tuple[Planform, str, bool]] = list()

for planform_param in planform_params:
    span = np.sqrt(wing_area * planform_param['aspect_ratio'])

    planforms.append((Planform(
        aspect_ratio=planform_param['aspect_ratio'],
        taper=planform_param['taper'],
        sweep_quarter_deg=planform_param['sweep'],
        thickness_to_chord=planform_param['thickness_to_chord'],
        cm_quarter_chord=planform_param['cm_ac'],
        cl0=planform_param['cl_0'],
        clmax=planform_param['cl_max'],
        flap=False, #NOTE for now
        airfoil_lift_slope=planform_param['cl_alpha'],
        wetted_surface_ratio=1.07,
        interference_factor=1.,
        span=span
    ), planform_param["pf_type"]))

In [5]:
print(planforms)

[(<Aircraft.Planform.Planform object at 0x000001E57F0120C0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E57F013470>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E57F012090>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E57F011A30>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E57F011FA0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E55073B9B0>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E55073BF20>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E55073A9C0>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E55073BF80>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E55073AA50>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E55073BB90>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E550738C50>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001E55073BFE0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001E550774080>, 'canard'), (<Aircraft.Planform

# Caching Planform properties

## CD0

In [6]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

for planform in planforms:
    planform[0].add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    planform[0].add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
    #NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    planform[0].add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    planform[0].add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

## Planform Mass

In [7]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

#TODO change to an actual structural weight estimation
for planform in planforms:
    size_planform(planform=planform[0], thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=0.33, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

Stresses 23197787.392732985, 63275972.13557633, 0.0004
Stresses 11498812.295544658, 32285322.880919207, 0.0007999999999999999
Stresses 7599153.92981523, 21966340.610221416, 0.0012
Stresses 23197787.392732985, 63275972.13557633, 0.0004
Stresses 11498812.295544658, 32285322.880919207, 0.0007999999999999999
Stresses 7599153.92981523, 21966340.610221416, 0.0012
Stresses 23674489.061909337, 63275972.13557633, 0.0004
Stresses 11737163.130132833, 32285322.880919207, 0.0007999999999999999
Stresses 7758054.486207346, 21966340.610221416, 0.0012
Stresses 23674489.061909337, 63275972.13557633, 0.0004
Stresses 11737163.130132833, 32285322.880919207, 0.0007999999999999999
Stresses 7758054.486207346, 21966340.610221416, 0.0012
Stresses 23197787.392732985, 63275972.13557633, 0.0004
Stresses 11498812.295544658, 32285322.880919218, 0.0007999999999999999
Stresses 7599153.92981523, 21966340.610221416, 0.0012
Stresses 23197787.392732985, 63275972.13557633, 0.0004
Stresses 11498812.295544658, 32285322.88091

# Saving the planforms

In [11]:
with open("pickles/planform_pickle_official.pcl", "w+b") as f:
    pickle.dump(planforms, f)

### Recovery to see if pickled correctly

In [12]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered = pickle.load(f)

In [13]:
sample:Planform = plaforms_recovered[5][0]
print(f"AR: {sample.aspect_ratio}")
print(f"cr: {sample.c_root}")
print(f"CD0 takeoff: {sample.CD0_cache["takeoff"]}")

pf_masses = [pf[0].mass_cache for pf in plaforms_recovered]
print(max(pf_masses), min(pf_masses))

AR: 5.0
cr: 0.23237900077244503
CD0 takeoff: 0.004301563921357677
0.3187671797469259 0.10048832625033727
